# Phase 4: Agent Orchestration & Multi-Agent Systems (Days 43–56)
## Day 45: Rebuild the ReAct loop inside LangGraph using conditional edges for tool execution

### 1. Core Theory (Just-in-Time)
**Why LangGraph for ReAct?**
The ReAct (Reason + Act) pattern relies on an LLM alternating between thinking about a problem and taking action (using tools). Traditional implementations often rely on `while` loops that are hard to interrupt, inspect, or manage state across turns. LangGraph treats the ReAct loop as a state machine. 

**How Conditional Edges Work:**
Instead of standard sequential flows, conditional edges evaluate the current state (e.g., 'Did the LLM invoke a tool?') and dynamically route to the next node (e.g., `execute_tools` vs `end_process`). This makes the loop explicit, observable, and strictly controlled.


In [1]:
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """
    The state of the agent.
    Keeps track of messages using the built-in add_messages reducer.
    """
    messages: Annotated[list[BaseMessage], add_messages]
    input: str


### 2. Code Implementation
Let's build a robust ReAct loop using `langgraph` and `langchain_core`. We will define a simple dummy tool, the LLM node, the tool execution node, and the critical **conditional edge** that decides the flow.


In [2]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode

# 1. Define a dummy tool with strict type hinting and docstrings
@tool
def get_weather(location: str) -> str:
    """Returns the weather for a given location."""
    return f"The weather in {location} is sunny and 75 degrees."

tools = [get_weather]

# Use the prebuilt ToolNode to safely execute tools in production
tool_node = ToolNode(tools)

# 2. Initialize the LLM (binding the tools so the LLM knows about them)
# Using ChatGroq as an example. In a real scenario, make sure GROQ_API_KEY is set.
llm = ChatGroq(model="llama3-8b-8192")
llm_with_tools = llm.bind_tools(tools)

# 3. Define the Nodes
def call_model(state: AgentState):
    """Invokes the LLM to decide the next step (Reason/Action)."""
    messages = state.get("messages", [])
    if not messages:
        messages = [HumanMessage(content=state["input"])]
    
    try:
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    except Exception as e:
        # Graceful fallback for local testing without API keys
        fallback_msg = AIMessage(content="API Error or missing key.")
        return {"messages": [fallback_msg]}

# 4. Define the Conditional Edge Logic
def should_continue(state: AgentState) -> str:
    """
    Determines whether to continue to tool execution or end the loop.
    Checks if the last message from the LLM contains tool calls.
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "continue"  # Route to tool execution
    
    return "end"  # Route to END


### 3. Orchestrating the Graph
Now we stitch the nodes together using the `StateGraph` and applying our conditional edge.

In [3]:
# Initialize the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# Set entry point
workflow.set_entry_point("agent")

# Add conditional edges from the 'agent' node
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "action",
        "end": END
    }
)

# Add standard edge from 'action' back to 'agent' to close the ReAct loop
workflow.add_edge("action", "agent")

# Compile the graph
app = workflow.compile()

print("LangGraph ReAct loop compiled successfully.")


LangGraph ReAct loop compiled successfully.


### 4. Practical Lab / Homework
**Task:** Expand the implementation above by adding a new tool called `calculate_sum` that takes two integers and returns their sum. Modify the graph invocation to test a prompt that requires both tools (e.g., 'What is the weather in Paris, and what is 5 + 7?').

**Instructions:**
1. Define the `@tool`.
2. Update the `tools` list.
3. Run the graph using `app.invoke()` with your prompt.


In [4]:
# LAB IMPLEMENTATION

@tool
def calculate_sum(a: int, b: int) -> int:
    """Calculates the sum of two integers."""
    return a + b

# NOTE: Re-run the graph compilation steps with the new tool list
lab_tools = [get_weather, calculate_sum]
lab_tool_node = ToolNode(lab_tools)
lab_llm_with_tools = llm.bind_tools(lab_tools)

def lab_call_model(state: AgentState):
    messages = state.get("messages", [])
    if not messages:
        messages = [HumanMessage(content=state["input"])]
    try:
        response = lab_llm_with_tools.invoke(messages)
        return {"messages": [response]}
    except Exception as e:
        return {"messages": [AIMessage(content="API Error or missing key during lab.")]}

lab_workflow = StateGraph(AgentState)
lab_workflow.add_node("agent", lab_call_model)
lab_workflow.add_node("action", lab_tool_node)
lab_workflow.set_entry_point("agent")
lab_workflow.add_conditional_edges("agent", should_continue, {"continue": "action", "end": END})
lab_workflow.add_edge("action", "agent")
lab_app = lab_workflow.compile()

# Execute (Wrapped in try/except for local execution without valid API keys)
try:
    final_state = lab_app.invoke({
        "input": "What is the weather in Paris, and what is 5 + 7?",
        "messages": []
    })
    for m in final_state['messages']:
        print(f"{m.type.upper()}: {m.content}")
except Exception as e:
    print("Execution halted (expected if GROQ_API_KEY is not set). Graph is structurally valid.")


AI: API Error or missing key during lab.


### 5. Common Pitfalls in Production
1. **Infinite Loops:** If the LLM continuously outputs tool calls that fail or don't answer the prompt, the ReAct loop will run endlessly. Always enforce a `recursion_limit` when calling `.invoke()` (e.g., `app.invoke(state, {"recursion_limit": 5})`).
2. **State Mutation Errors:** In LangGraph, state updates are additive or overwritten based on the `Annotated` reducers (like `operator.add` or `add_messages`). Misconfiguring the reducer often results in swallowed messages or duplicated context.
3. **Tool Hallucinations:** LLMs might invent tool names or pass incorrect arguments. Robust ToolNodes must handle validation errors and feed the error *back* to the LLM so it can retry.
